In [16]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"
%matplotlib inline

In [50]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other , Value)  else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad 
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int , float)),  "only support int/float for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out
    
    def __mul__(self, other):
        other = other if isinstance(other , Value)  else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad 
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __sub__(self, other):
        return self + (-other)
    
    def __truediv__(self, other):
        return self * other**-1
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):
        return self + other
    
    def __neg__(self):
        return self * -1 
    

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out
    
    def backward(self):
        topo = []

        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1
        for node in reversed(topo):
            node._backward()


In [25]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name = uid + n._op, label = n._op)
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot


In [ ]:
#inputs x1,x2
x1 = Value(2, label='x1')
x2 = Value(0, label ='x2')
#weights w1,w2
w1 = Value(-3, label='w1')
w2 = Value(1, label='w2')
#bias of the neuron
b = Value(6.8813735, label='b')
# x1w1 + x2w2 + b
x1w1 = x1 * w1; x1w1.label = 'x1 * w1'
x2w2 = x2 * w2; x2w2.label = 'x2 * w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1 + x2w2'
n = x1w1x2w2 + b; n.label = 'n'
#-----
e = (2*n).exp() ; e.label = 'e'
o = (e - 1) / (e + 1) ; o.label = "o"
o.backward()

draw_dot(o)

In [ ]:
import torch

In [70]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
        
    def __call__(self, x):
        # w * x + b 
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        
        return out
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]
        
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)   
        return x
    
    def parameters(self):
        return [p for layers in self.layers for p in layers.parameters()]
    
 

In [92]:
x = [2.0, 3.0, -1]
n = MLP(3, [4, 4 ,1])

n(x)

Value(data=0.4850083714095647)

In [73]:
n.parameters()

[Value(data=0.05049325738497856),
 Value(data=-0.9801267385368615),
 Value(data=-0.7373908942199263),
 Value(data=-0.56400605955957),
 Value(data=0.3212040332668249),
 Value(data=0.9680272091797748),
 Value(data=-0.7339058224645907),
 Value(data=-0.6158462437273795),
 Value(data=-0.33223965922351306),
 Value(data=-0.7011127927265604),
 Value(data=0.08820043650545273),
 Value(data=-0.8730823385599482),
 Value(data=0.39563664747718774),
 Value(data=0.9284552623370776),
 Value(data=-0.7664359389311779),
 Value(data=0.6748724229023189),
 Value(data=0.749935451878303),
 Value(data=0.13120575894144326),
 Value(data=0.5173566715566811),
 Value(data=0.7059721154595928),
 Value(data=0.1821294345778368),
 Value(data=0.4840561110769981),
 Value(data=0.9499952990684248),
 Value(data=-0.26222493104399125),
 Value(data=0.8433297030375861),
 Value(data=-0.6021252807337232),
 Value(data=0.829004310663247),
 Value(data=-0.9831637626672862),
 Value(data=-0.9618123725288239),
 Value(data=-0.7519421159358

In [90]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]


In [93]:
for k in range(20):
    #forward
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    #backward       
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()
    
    #update
    for p in n.parameters():
        p.data += -0.05 * p.grad
        
    print(k, loss.data)

0 4.717218160615458
1 3.5774853077495985
2 1.822685272760211
3 0.849535763921584
4 0.3464034221434005
5 0.2743781834053436
6 0.22682527067100322
7 0.19193530581672993
8 0.16535964028927141
9 0.1445534995829561
10 0.1278981925358743
11 0.11431653837306896
12 0.10306632183087996
13 0.09362109999397045
14 0.0855981342296944
15 0.07871315626539521
16 0.0727510596601792
17 0.06754637750973339
18 0.06296995862688365
19 0.058919675941139574


In [94]:
ypred

[Value(data=0.95063182146629),
 Value(data=-0.8959922201653575),
 Value(data=-0.853280883647541),
 Value(data=0.8446348123934276)]